# METS-R SIM Advanced Commands

This notebook walks through METSRClient APIs that are useful for interactive experiments: co-simulation control, dynamic facilities and fleets, ride-hailing commands, Kafka/V2X streaming, save/load, and cleanup. The examples favor direct client calls so the request and response patterns are easy to reuse in your own scripts.

## Setup

Run this cell first. Keep the flags set to `False` while reading, then enable only the section you want to execute. Most sections start a simulator instance, so run one section at a time unless you intentionally want multiple clients open.

In [ ]:
import os
import sys
import time
import math
import random
import shutil
from collections import Counter
from pathlib import Path
from statistics import mean

import pandas as pd
import numpy as np
from tqdm import tqdm

if os.path.basename(os.getcwd()) == "tutorials":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from clients.METSRClient import METSRClient
from clients.KafkaDataProcessor import KafkaDataProcessor
from utils.util import read_run_config, prepare_sim_dirs, run_simulation_in_docker
from utils.util import clear_all
clear_all() # clear all mets-r related processes, if any


# Switches to control which cells to run.
RUN_COSIM_CELLS = True
RUN_FACILITY_CELLS = False
RUN_RIDEHAILING_CELLS = False
RUN_LONG_EXPERIMENTS = False
RUN_KAFKA_CELLS = False

print("Working directory:", os.getcwd())

## Running a Section

Each section reads a config, prepares simulation output folders, starts METS-R SIM in Docker, and then constructs a `METSRClient` directly. The repeated setup is intentional: the notebook is meant to show the client APIs in the same shape you would use from a standalone script.

When a section finishes, terminate its client before starting another section. The final cleanup section is a safety net for any clients that are still open.

# 1. Co-Simulation Commands

Use these calls to register co-simulation roads, inspect queued vehicles, move vehicles onto controlled roads, update routes, query route alternatives, release roads, and terminate the client.

In [ ]:
if RUN_COSIM_CELLS:
    cosim_config = read_run_config("configs/run_cosim_CARLAT5.json")
    cosim_sim_dirs = prepare_sim_dirs(cosim_config)
    run_simulation_in_docker(cosim_config)

    cosim_client = METSRClient(
        host=cosim_config.metsr_host,
        sim_folder=cosim_sim_dirs[0],
        port=cosim_config.ports[0] if hasattr(cosim_config, "ports") else cosim_config.metsr_port,
        verbose=False,
        timeout=120,
    )
else:
    cosim_config = cosim_client = cosim_sim_dirs = None

In [ ]:
cosim_client.query_route_between_roads("-47", "40")

In [ ]:
COSIM_ROADS_TOWN05 = ["-47", "17", "-1", "1", "-0", "0", "40", "-18"]
cosim_registration_result = None

if RUN_COSIM_CELLS and cosim_client is not None:
    set_response = cosim_client.set_cosim_road(["-47"])

    trip_responses = []
    for vehicle_id in range(3):
        trip_responses.append(cosim_client.generate_trip_between_roads(vehicle_id, "-47", "40"))

    cosim_client.tick(1)
    queue_response = cosim_client.query_coSimVehicle()

    cosim_registration_result = {
        "set_cosim_road": set_response,
        "generate_trip_between_roads": trip_responses,
        "query_coSimVehicle": queue_response,
    }

cosim_registration_result

In [ ]:
cosim_connection_result = None

if RUN_COSIM_CELLS and cosim_client is not None:
    cosim_client.set_cosim_road(["0", "-47"])

    cosim_connection_result = {
        "tick": cosim_client.current_tick,
        "centerline": cosim_client.query_centerline("-47", transform_coords=True),
        "signal": cosim_client.query_signal_between_roads("-47", "0"),
        "route_options": cosim_client.query_route_between_roads("-47", "0"),
        "cosim_vehicles": cosim_client.query_coSimVehicle(),
    }

cosim_connection_result

Vehicles queued inside a co-simulation region do not enter the road automatically. After the external simulator is ready for a vehicle, call `enter_road_from_queue`. Vehicles outside co-simulation regions still enter roads through the normal simulator logic.

In [ ]:
cosim_entry_result = None

if RUN_COSIM_CELLS and cosim_client is not None:
    enter_response = cosim_client.enter_road_from_queue(0, "-47", private_veh=True)
    cosim_client.tick(1)

    cosim_entry_result = {
        "enter_road_from_queue": enter_response,
        "query_vehicle": cosim_client.query_vehicle(0, private_veh=True),
        "query_coSimVehicle": cosim_client.query_coSimVehicle(),
    }

cosim_entry_result

In case you would like to spawn a vehicle immediately. Use teleport_trace_replay_vehicle to make vehicle enter a specific road, if it is in a cosim region, you can use teleport_cosim_vehicle to teleport it to a specific location.

Note that purely teleport_trace_replay_vehicle projects the provided x, y coordinates onto the centerline of the target lane since in METS-R all vehicles are running along with the lane centerlines, the final vehicle position may differ from the original x, y values provided by the user.

In [ ]:
spawn_entery_result = None

if RUN_COSIM_CELLS and cosim_client is not None:
    spawn_response = cosim_client.generate_trip_between_roads(222, "-47", "40")
    teleport_response = cosim_client.teleport_trace_replay_vehicle(222, roadID = "-47", laneID = 0, x = 0, y = 0, private_veh=True)
    teleport_response2 = cosim_client.teleport_cosim_vehicle(222, x = 0, y = 0, z= 0, bearing = 0, private_veh=True) # METS-R won't check the validity of the x, y, z values, so be careful
    spawn_entery_result = {
        "spawn_vehicle_on_road": spawn_response,
        'teleport_trace_replay_vehicle': teleport_response,
        "teleport_cosim_vehicle": teleport_response2,
        "query_vehicle": cosim_client.query_vehicle(222, private_veh=True),
        "query_coSimVehicle": cosim_client.query_coSimVehicle(),
    }

spawn_entery_result

A vehicle route can be replaced with a full road sequence or with intermediate roads. METS-R SIM fills in the missing road-to-road path when it can find one.

In [ ]:
route_update_result = None

if RUN_COSIM_CELLS and cosim_client is not None:
    route_response = cosim_client.update_vehicle_route(0, ["-47", "17", "40"], private_veh=True)

    route_update_result = {
        "update_vehicle_route": route_response,
        "query_coSimVehicle": cosim_client.query_coSimVehicle(),
    }

route_update_result

In [ ]:
route_candidates = None

if RUN_COSIM_CELLS and cosim_client is not None:
    route_candidates = cosim_client.query_k_routes_between_roads("-47", "0", k=3)

route_candidates

In [ ]:
cosim_release_result = None

if RUN_COSIM_CELLS and cosim_client is not None:
    release_response = cosim_client.release_cosim_road(COSIM_ROADS_TOWN05)
    cosim_client.tick(1)
    cosim_client.terminate()
    cosim_client = None

    cosim_release_result = {"release_cosim_road": release_response}

cosim_release_result

# 2. Facilities, Fleets, and Charging

This section creates a zone, a charging station, taxis, and a bus route, then sends taxis to charge. The returned IDs are used immediately in follow-up queries so you can see how add/query/control calls fit together.

Every simulation needs at least one zone. If a config omits them, METS-R SIM creates temporary defaults near the average road coordinates. Once you add real zones, the temporary defaults are removed automatically.

In [ ]:
if RUN_FACILITY_CELLS:
    facility_config = read_run_config("configs/run_cosim_CARLAT5.json")
    facility_sim_dirs = prepare_sim_dirs(facility_config)
    run_simulation_in_docker(facility_config)

    facility_client = METSRClient(
        host=facility_config.metsr_host,
        sim_folder=facility_sim_dirs[0],
        port=facility_config.ports[0] if hasattr(facility_config, "ports") else facility_config.metsr_port,
        verbose=False,
        timeout=120
    )

In [ ]:
facility_result = None

if RUN_FACILITY_CELLS:
      facility_result = {
        "query_zone": facility_client.query_zone(),
        "query_chargingStation": facility_client.query_chargingStation(),
        "query_taxi": facility_client.query_taxi(),
        "bus_ids": facility_client.query_bus(),
    }
else:
      faclity_result = None
      
facility_result

In [ ]:
facility_client.query_taxi(0)

In [ ]:
facility_result = None


if RUN_FACILITY_CELLS:
    ref_zone = facility_client.query_zone(0)["DATA"][0]

    zone_response = facility_client.add_zone(
        ref_zone["x"] + 0.0001,
        ref_zone["y"] + 0.0001,
        capacity=-1,
        zone_type=0,
    )
    new_zone_id = zone_response["DATA"][0]["ID"]

    zone_response2 = facility_client.add_zone(
        ref_zone["x"] - 0.0001,
        ref_zone["y"] - 0.0001,
        capacity=-1,
        zone_type=0,
    )
    new_zone_id2 = zone_response2["DATA"][0]["ID"]

    station_response = facility_client.add_charging_station(
        ref_zone["x"] + 0.0001,
        ref_zone["y"] + 0.0001,
        num_l2=2,
        num_l3=1,
        num_bus=0,
        price_l2=0.30,
        price_l3=0.50,
    )
    station_id = station_response["DATA"][0]["ID"]

    taxi_response = facility_client.add_taxi(zoneID=new_zone_id, num=3)
    taxi_ids = taxi_response["DATA"][0]["IDs"]

    bus_route_response = facility_client.add_bus_route("debug_route", zone=[1, 2], road=["-0", "-2"])
    bus_response = facility_client.add_bus(routeName="debug_route", num=2)
    bus_ids = bus_response["DATA"][0]["IDs"]

    station_charge_response = facility_client.go_charging(taxi_ids[1], veh_type=False, charger_type=0, cs_id=station_id)

    facility_result = {
        "add_zone": zone_response,
        "add_charging_station": station_response,
        "add_taxi": taxi_response,
        "add_bus_route": bus_route_response,
        "add_bus": bus_response,
        "go_charging_specific_station": station_charge_response,
        "query_zone": facility_client.query_zone(new_zone_id),
        "query_chargingStation": facility_client.query_chargingStation(station_id),
        "query_taxi": facility_client.query_taxi(taxi_ids),
        "bus_ids": bus_ids,
    }

    
else:
    facility_config = facility_client = facility_sim_dirs = None

facility_result

In [ ]:
if RUN_FACILITY_CELLS and facility_client is not None:
      facility_client.terminate()

# 3. Ride-Hailing Control Commands

These cells use a controlled NYC configuration: background ride-hailing demand is disabled, and request creation, dispatching, repositioning, and charging are controlled through METSRClient calls.

In [ ]:
TAXI_STATES = {
    -1: "NONE",
    0: "PARKING",
    1: "OCCUPIED_TRIP",
    2: "INACCESSIBLE_RELOCATION_TRIP",
    4: "CHARGING_TRIP",
    5: "CRUISING_TRIP",
    6: "PICKUP_TRIP",
    7: "ACCESSIBLE_RELOCATION_TRIP",
    9: "CHARGING_RETURN_TRIP",
}
LEFT_ZONE_FIELDS = ("leftTaxiRequests", "leftTaxiPassengers", "leftBusRequests", "leftBusPassengers")

if RUN_RIDEHAILING_CELLS:
    rh_config = read_run_config("configs/run_interactive_NYC.json")
    rh_config.name = "NYC_Ridehailing_Advanced"
    rh_config.rh_demand_file = None
    rh_config.rh_wait_file = None
    rh_config.rh_share_file = None
    rh_config.rh_demand_factor = 0.0
    rh_config.num_etaxi = 100
    rh_config.num_ebus = 0
    rh_config.dispatching_controlled_by_control_apis = True
    rh_config.repositioning_controlled_by_control_apis = True
    rh_config.charging_controlled_by_control_apis = True
    rh_config.json_output = False

    rh_sim_dirs = prepare_sim_dirs(rh_config)
    run_simulation_in_docker(rh_config)

    rh_client = METSRClient(
        host=rh_config.metsr_host,
        sim_folder=rh_sim_dirs[0],
        port=rh_config.ports[0] if hasattr(rh_config, "ports") else rh_config.metsr_port,
        timeout=300,
    )
else:
    rh_config = rh_client = rh_sim_dirs = None

In [ ]:
request_result = None

if RUN_RIDEHAILING_CELLS and rh_client is not None:
    zone_ids = list(rh_client.query_zone().get("id_list", []))
    origin_zone, dest_zone = zone_ids[0], zone_ids[1]

    origin_before = rh_client.query_zone(origin_zone)["DATA"][0]
    request_response = rh_client.add_taxi_requests(origin_zone, dest_zone, 3)
    req_id = next(
        (item["reqID"] for item in request_response.get("DATA", []) if item.get("STATUS", "OK") == "OK" and item.get("reqID") is not None),
        None,
    )

    rh_client.tick(max(1, int(round(65 / rh_config.sim_step_size))), wait_forever=True)
    origin_after = rh_client.query_zone(origin_zone)["DATA"][0]

    request_result = {
        "add_taxi_requests": request_response,
        "query_request": rh_client.query_request(req_id) if req_id is not None else None,
        "origin_zone": origin_zone,
        "dest_zone": dest_zone,
        "taxi_demand_before": origin_before.get("taxi_demand"),
        "taxi_demand_after": origin_after.get("taxi_demand"),
        "left_request_deltas": {
            field: origin_after.get(field, 0) - origin_before.get(field, 0)
            for field in LEFT_ZONE_FIELDS
        },
    }

request_result

In [ ]:
dispatch_reposition_result = None

if RUN_RIDEHAILING_CELLS and rh_client is not None:
    available_taxis = rh_client.query_available_taxis().get("DATA", [])
    if available_taxis:
        taxi_id = available_taxis[0]["ID"]
    else:
        zone_ids = list(rh_client.query_zone().get("id_list", []))
        taxi_id = rh_client.add_taxi(zone_ids[0], 1)["DATA"][0]["IDs"][0]

    taxi_before = rh_client.query_taxi(taxi_id)["DATA"][0]
    zone_ids = list(rh_client.query_zone().get("id_list", []))
    origin_zone = taxi_before.get("origin") or zone_ids[0]
    dest_zone = next(zone_id for zone_id in zone_ids if zone_id != origin_zone)

    request_response = rh_client.add_taxi_requests(origin_zone, dest_zone, 2)
    req_id = next(
        (item["reqID"] for item in request_response.get("DATA", []) if item.get("STATUS", "OK") == "OK" and item.get("reqID") is not None),
        None,
    )

    dispatch_response = rh_client.dispatch_taxi(taxi_id, req_id) if req_id is not None else None
    taxi_after_dispatch = rh_client.query_taxi(taxi_id)["DATA"][0]
    rh_client.tick(max(1, int(round(30 / rh_config.sim_step_size))), wait_forever=True)
    taxi_after_settle = rh_client.query_taxi(taxi_id)["DATA"][0]

    available_after_dispatch = rh_client.query_available_taxis().get("DATA", [])
    reposition_taxi_id = available_after_dispatch[0]["ID"] if available_after_dispatch else rh_client.add_taxi(origin_zone, 1)["DATA"][0]["IDs"][0]
    reposition_before = rh_client.query_taxi(reposition_taxi_id)["DATA"][0]
    target_zone = next(zone_id for zone_id in zone_ids if zone_id != reposition_before.get("origin"))
    reposition_response = rh_client.reposition_taxi(reposition_taxi_id, target_zone)
    rh_client.tick(max(1, int(round(180 / rh_config.sim_step_size))), wait_forever=True)
    reposition_after = rh_client.query_taxi(reposition_taxi_id)["DATA"][0]

    dispatch_reposition_result = {
        "dispatch_taxi": dispatch_response,
        "dispatch_taxi_id": taxi_id,
        "dispatch_req_id": req_id,
        "dispatch_state_before": TAXI_STATES.get(taxi_before.get("state"), taxi_before.get("state")),
        "dispatch_state_after_call": TAXI_STATES.get(taxi_after_dispatch.get("state"), taxi_after_dispatch.get("state")),
        "dispatch_state_after_settle": TAXI_STATES.get(taxi_after_settle.get("state"), taxi_after_settle.get("state")),
        "reposition_taxi": reposition_response,
        "reposition_taxi_id": reposition_taxi_id,
        "reposition_target_zone": target_zone,
        "reposition_state_before": TAXI_STATES.get(reposition_before.get("state"), reposition_before.get("state")),
        "reposition_state_after": TAXI_STATES.get(reposition_after.get("state"), reposition_after.get("state")),
    }

dispatch_reposition_result

In [ ]:
charging_result = None

if RUN_RIDEHAILING_CELLS and rh_client is not None:
    available_taxis = rh_client.query_available_taxis().get("DATA", [])
    taxi_id = available_taxis[0]["ID"] if available_taxis else rh_client.add_taxi(1, 1)["DATA"][0]["IDs"][0]
    taxi_before = rh_client.query_taxi(taxi_id)["DATA"][0]
    station_zone = taxi_before.get("origin") or list(rh_client.query_zone().get("id_list", []))[0]
    zone = rh_client.query_zone(station_zone)["DATA"][0]

    station_response = rh_client.add_charging_station(
        zone["x"],
        zone["y"],
        num_l2=2,
        num_l3=1,
        num_bus=0,
        price_l2=0.30,
        price_l3=0.50,
    )
    station_id = station_response["DATA"][0]["ID"]

    charge_response = rh_client.go_charging(taxi_id, veh_type=False, charger_type=0, cs_id=station_id)
    taxi_samples = []
    station_samples = []
    for _ in range(4):
        rh_client.tick(int(round(300 / rh_config.sim_step_size)), wait_forever=True)
        taxi_samples.append(rh_client.query_taxi(taxi_id)["DATA"][0])
        station_samples.append(rh_client.query_chargingStation(station_id)["DATA"][0])

    charging_result = {
        "add_charging_station": station_response,
        "go_charging": charge_response,
        "taxi_id": taxi_id,
        "station_id": station_id,
        "taxi_state_path": [TAXI_STATES.get(sample.get("state"), sample.get("state")) for sample in taxi_samples],
        "station_busy_l2": [sample.get("l2_charger", 0) - sample.get("num_available_l2", 0) for sample in station_samples],
        "station_busy_dcfc": [sample.get("dcfc_charger", 0) - sample.get("num_available_dcfc", 0) for sample in station_samples],
    }

charging_result

In [ ]:
if RUN_RIDEHAILING_CELLS and rh_client is not None:
    rh_client.terminate()

# 4. Matcher Comparison with Save / Load

Preheat the simulator for 30 minutes with a fixed fleet and request rate so all matchers start from the same realistic traffic state. The preheated state is saved with `save`, then loaded once per matching algorithm so each run begins from identical conditions. Only the dispatching algorithm varies between runs.

In [ ]:
PREHEAT_MINUTES = 30
EXPERIMENT_MINUTES = 30
SAMPLE_EVERY_SECONDS = 60
RANDOM_SEED = 20260514
PRACTICAL_TIMEOUT = 300
FLEET_SIZE = 100
REQUEST_RATE_PER_MIN = 5
MATCHERS = ["fcfs", "greedy_nearest"]
LEFT_ZONE_FIELDS = ("leftTaxiRequests", "leftTaxiPassengers", "leftBusRequests", "leftBusPassengers")

In [ ]:
exp_client = None
exp_config = None
copied_save_path = None
save_response = None

if RUN_LONG_EXPERIMENTS:
    exp_config = read_run_config("configs/run_interactive_NYC.json")
    exp_config.name = "RH_Preheat_And_Experiment"
    exp_config.rh_demand_file = None
    exp_config.rh_wait_file = None
    exp_config.rh_share_file = None
    exp_config.rh_demand_factor = 0.0
    exp_config.num_etaxi = FLEET_SIZE
    exp_config.num_ebus = 0
    exp_config.dispatching_controlled_by_control_apis = True
    exp_config.repositioning_controlled_by_control_apis = True
    exp_config.charging_controlled_by_control_apis = True
    exp_config.json_output = False

    exp_sim_dirs = prepare_sim_dirs(exp_config)
    run_simulation_in_docker(exp_config)
    exp_client = METSRClient(
        host=exp_config.metsr_host,
        sim_folder=exp_sim_dirs[0],
        port=exp_config.ports[0] if hasattr(exp_config, "ports") else exp_config.metsr_port,
        timeout=PRACTICAL_TIMEOUT,
    )

    rng = random.Random(RANDOM_SEED)
    all_zone_ids = list(exp_client.query_zone().get("id_list", []))
    zone_ids = all_zone_ids[:min(6, len(all_zone_ids))]

    print(f"Preheating for {PREHEAT_MINUTES} minutes ...")
    for _ in tqdm(range(PREHEAT_MINUTES)):
        origins = [rng.choice(zone_ids) for _ in range(REQUEST_RATE_PER_MIN)]
        destinations = [rng.choice([z for z in zone_ids if z != o]) for o in origins]
        passengers = [rng.randint(1, 3) for _ in range(REQUEST_RATE_PER_MIN)]

        exp_client.add_taxi_requests(origins, destinations, passengers)
        pending = sorted(
            [r for r in exp_client.query_pending_requests().get("DATA", [])
             if str(r.get("status", "")).startswith("pending_taxi") and r.get("STATUS", "OK") == "OK"],
            key=lambda r: (r.get("generationTime", 0), r.get("ID", 0)),
        )
        available = sorted(
            [t for t in exp_client.query_available_taxis().get("DATA", []) if t.get("ID") is not None],
            key=lambda t: t.get("ID", 0),
        )
        matches = [{"taxi_id": t["ID"], "req_id": r["ID"]} for t, r in zip(available, pending)]
        if matches:
            exp_client.dispatch_taxi([m["taxi_id"] for m in matches], [m["req_id"] for m in matches])
        exp_client.tick(
            max(1, int(round(SAMPLE_EVERY_SECONDS / exp_config.sim_step_size))),
            wait_forever=True,
        )

    save_name = "preheat_state.zip"
    save_response = exp_client.save(save_name)

    # If you need to run the experiment in multiple instance, you can copy the saved state from the latest output folder to a known location and load from there in the next instance.
    # output_root = Path("output")
    # latest_output = max([p for p in output_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)
    # saved_path = latest_output / save_name
    # copied_save_path = save_name
    # copied_save_path.parent.mkdir(exist_ok=True)
    # if saved_path.exists():
    #     shutil.copy(saved_path, copied_save_path)

    # print("Preheat complete. State saved to:", copied_save_path)

save_response

In [ ]:
practical_results = []

if RUN_LONG_EXPERIMENTS and exp_client is not None:
    try:
        for matcher_name in MATCHERS:
            exp_client.load(save_name, reload_network=False)

            rng = random.Random(RANDOM_SEED)
            all_zone_ids = list(exp_client.query_zone().get("id_list", []))
            zone_ids = all_zone_ids[:min(6, len(all_zone_ids))]
            generated_req_ids = []
            generated_passengers = 0
            matched_req_ids = []

            print(f"Running matcher={matcher_name} for {EXPERIMENT_MINUTES} minutes ...")
            for _ in tqdm(range(EXPERIMENT_MINUTES)):
                origins = [rng.choice(zone_ids) for _ in range(REQUEST_RATE_PER_MIN)]
                destinations = [rng.choice([z for z in zone_ids if z != o]) for o in origins]
                passengers = [rng.randint(1, 3) for _ in range(REQUEST_RATE_PER_MIN)]

                req_resp = exp_client.add_taxi_requests(origins, destinations, passengers)
                accepted = [
                    (item, p)
                    for item, p in zip(req_resp.get("DATA", []), passengers)
                    if item.get("STATUS", "OK") == "OK" and item.get("reqID") is not None
                ]
                generated_req_ids.extend(item["reqID"] for item, _ in accepted)
                generated_passengers += sum(p for _, p in accepted)

                pending = [
                    r for r in exp_client.query_pending_requests().get("DATA", [])
                    if str(r.get("status", "")).startswith("pending_taxi") and r.get("STATUS", "OK") == "OK"
                ]
                available = [t for t in exp_client.query_available_taxis().get("DATA", []) if t.get("ID") is not None]

                if matcher_name == "greedy_nearest" and available and pending:
                    zone_cache = {}
                    candidates = []
                    for taxi in available:
                        tx, ty = taxi.get("x"), taxi.get("y")
                        if tx is None or ty is None:
                            continue
                        for req in pending:
                            oz = req.get("origin")
                            if oz not in zone_cache:
                                zone_cache[oz] = exp_client.query_zone(oz)["DATA"][0]
                            z = zone_cache[oz]
                            zx, zy = z.get("x"), z.get("y")
                            if zx is not None and zy is not None:
                                candidates.append((math.hypot(tx - zx, ty - zy), taxi["ID"], req["ID"]))
                    used_taxis, used_reqs, matches = set(), set(), []
                    for _, tid, rid in sorted(candidates):
                        if tid not in used_taxis and rid not in used_reqs:
                            matches.append({"taxi_id": tid, "req_id": rid})
                            used_taxis.add(tid)
                            used_reqs.add(rid)
                else:
                    available = sorted(available, key=lambda t: t.get("ID", 0))
                    pending = sorted(pending, key=lambda r: (r.get("generationTime", 0), r.get("ID", 0)))
                    matches = [{"taxi_id": t["ID"], "req_id": r["ID"]} for t, r in zip(available, pending)]

                if matches:
                    disp = exp_client.dispatch_taxi(
                        [m["taxi_id"] for m in matches],
                        [m["req_id"] for m in matches],
                    )
                    matched_req_ids.extend(
                        item.get("reqID")
                        for item in disp.get("DATA", [])
                        if item.get("STATUS") == "OK" and item.get("reqID") is not None
                    )

                exp_client.tick(
                    max(1, int(round(SAMPLE_EVERY_SECONDS / exp_config.sim_step_size))),
                    wait_forever=True,
                )

            request_records = exp_client.query_request(generated_req_ids).get("DATA", []) if generated_req_ids else []
            left_totals = {field: 0 for field in LEFT_ZONE_FIELDS}
            for zone in exp_client.query_zone(zone_ids).get("DATA", []):
                for field in LEFT_ZONE_FIELDS:
                    left_totals[field] += zone.get(field, 0)

            taxi_ids = exp_client.query_taxi().get("id_list", [])[:300]
            taxi_records = exp_client.query_taxi(taxi_ids).get("DATA", []) if taxi_ids else []
            state_counts = Counter(taxi.get("state") for taxi in taxi_records)
            pending_end = exp_client.query_pending_requests().get("DATA", [])
            available_end = exp_client.query_available_taxis().get("DATA", [])

            match_waits = [
                (r.get("matchedTime") - r.get("generationTime")) * exp_config.sim_step_size
                for r in request_records
                if r.get("generationTime") is not None and r.get("matchedTime") is not None and r.get("matchedTime") >= 0
            ]
            pickup_waits = [
                (r.get("pickupTime") - r.get("generationTime")) * exp_config.sim_step_size
                for r in request_records
                if r.get("generationTime") is not None and r.get("pickupTime") is not None and r.get("pickupTime") >= 0
            ]

            practical_results.append({
                "matcher": matcher_name,
                "generated_requests": len(generated_req_ids),
                "generated_passengers": generated_passengers,
                "matched_requests": len(matched_req_ids),
                "match_rate": len(matched_req_ids) / len(generated_req_ids) if generated_req_ids else None,
                "left_taxi_requests": left_totals.get("leftTaxiRequests", 0),
                "left_taxi_passengers": left_totals.get("leftTaxiPassengers", 0),
                "remaining_pending": len(pending_end),
                "available_taxis_end": len(available_end),
                "pickup_state_taxis_end": state_counts.get(6, 0),
                "occupied_taxis_end": state_counts.get(1, 0),
                "mean_match_wait_seconds": mean(match_waits) if match_waits else None,
                "mean_pickup_wait_seconds": mean(pickup_waits) if pickup_waits else None,
            })
    finally:
        exp_client.terminate()
        exp_client = None
        # copied_save_path.unlink(missing_ok=True) # if you copied the save file out, use this to remove it in the end.

In [ ]:
pd.DataFrame(practical_results) if practical_results else None

# 5. Kafka and V2X Sensor Stream Commands

This section starts Kafka, runs a Town05 co-simulation config, generates vehicles, promotes a subset to V2X sensors, advances the simulator, and reads the first BSM batch from the Kafka processor.

In [ ]:
kafka_config = kafka_client = kafka_processor = kafka_sim_dirs = None
first_bsm_batch = None

if RUN_KAFKA_CELLS:
    cwd = os.getcwd()
    try:
        os.chdir("docker")
        os.system("docker-compose up -d")
    finally:
        os.chdir(cwd)

    time.sleep(10)

    kafka_config = read_run_config("configs/run_cosim_CARLAT5.json")
    kafka_sim_dirs = prepare_sim_dirs(kafka_config)
    run_simulation_in_docker(kafka_config)

    kafka_client = METSRClient(
        host=kafka_config.metsr_host,
        sim_folder=kafka_sim_dirs[0],
        port=kafka_config.ports[0] if hasattr(kafka_config, "ports") else kafka_config.metsr_port,
        timeout=120,
    )
    kafka_processor = KafkaDataProcessor(kafka_config)

    kafka_client.generate_trip(list(range(100)), -1, -1)
    kafka_client.update_vehicle_sensor_type(list(range(10, 20)), 1, True)

    for _ in range(100):
        kafka_client.tick(10)
        first_bsm_batch = kafka_processor.process()
        if first_bsm_batch is not None:
            break

first_bsm_batch

# 6. Cleanup Commands

Run this after any section that starts a client. Kafka cleanup also stops the Docker Compose services in `docker/`.

In [ ]:
for name in ["cosim_client", "facility_client", "rh_client", "kafka_client", "exp_client"]:
    client = globals().get(name)
    if client is not None:
        print("Terminating", name)
        try:
            client.terminate()
        except Exception as exc:
            print(f"Could not terminate {name}: {exc}")
        globals()[name] = None

if RUN_KAFKA_CELLS:
    cwd = os.getcwd()
    try:
        os.chdir("docker")
        os.system("docker-compose down")
    finally:
        os.chdir(cwd)